# 2. ref

# 2. ref

## A. kaggle

ref: <https://www.kaggle.com/c/aerial-cactus-identification>

## B. 데이터 살펴보기

## C. 제출하기

# 3. imports

In [42]:
import datasets
import transformers
import torchvision.transforms
import evaluate
import numpy as np
import pandas as pd 
import PIL.Image
import torch

# 4. 예비학습

# 5. Step1~2

## A. 압축풀기

In [3]:
import os
import zipfile

# 'data' 폴더가 없으면 생성
if not os.path.exists('./data'):
    os.makedirs('data')

# 압축 파일 풀기
with zipfile.ZipFile('aerial-cactus-identification.zip', 'r') as zip_ref:  # '파일명.zip'을 실제 파일명으로 변경
    zip_ref.extractall('./data')

with zipfile.ZipFile('./data/test.zip', 'r') as zip_ref:
    zip_ref.extractall('./data')    

with zipfile.ZipFile('./data/train.zip', 'r') as zip_ref:
    zip_ref.extractall('./data')    

## B. 모델

In [5]:
model = transformers.AutoModelForImageClassification.from_pretrained(
    "google/vit-base-patch16-224-in21k",
    num_labels=2,
)

Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.

## C. 샘플자료

In [33]:
cat ./data/train.csv | head -n 5

id,has_cactus
0004be2cfeaba1c0361d39e2b000257b.jpg,1
000c8a36845c0208e833c79c1bffedd1.jpg,1
000d1e9a533f62e55c289303b072733d.jpg,1
0011485b40695e9138e92d0b3fb55128.jpg,1
cat: write error: Broken pipe

-   마지막은 오류는 무시해도 상관없음..

In [34]:
img1 = PIL.Image.open('./data/train/0004be2cfeaba1c0361d39e2b000257b.jpg')
img2 = PIL.Image.open('./data/train/000c8a36845c0208e833c79c1bffedd1.jpg')
label1 = 1 
label2 = 1 

In [51]:
f1 = torchvision.transforms.ToTensor()
f1(img1)

tensor([[[0.5333, 0.5255, 0.5765,  ..., 0.5961, 0.5765, 0.6157],
         [0.4863, 0.6314, 0.6196,  ..., 0.6392, 0.5765, 0.5686],
         [0.6314, 0.6275, 0.6196,  ..., 0.5882, 0.6314, 0.5059],
         ...,
         [0.5373, 0.7725, 0.4314,  ..., 0.6235, 0.5686, 0.6118],
         [0.5882, 0.6627, 0.5882,  ..., 0.6549, 0.5961, 0.5961],
         [0.7176, 0.4510, 0.6471,  ..., 0.7020, 0.5569, 0.5451]],

        [[0.5412, 0.5333, 0.5804,  ..., 0.5059, 0.4863, 0.5255],
         [0.4941, 0.6392, 0.6235,  ..., 0.5490, 0.4941, 0.4863],
         [0.6392, 0.6353, 0.6235,  ..., 0.5059, 0.5490, 0.4235],
         ...,
         [0.4353, 0.6706, 0.3294,  ..., 0.5176, 0.4627, 0.5059],
         [0.4863, 0.5608, 0.4863,  ..., 0.5490, 0.4824, 0.4824],
         [0.6157, 0.3490, 0.5451,  ..., 0.5882, 0.4431, 0.4314]],

        [[0.4902, 0.4902, 0.5490,  ..., 0.5294, 0.5098, 0.5490],
         [0.4431, 0.5961, 0.5922,  ..., 0.5725, 0.5137, 0.5059],
         [0.5882, 0.5922, 0.5922,  ..., 0.5255, 0.5686, 0.

In [56]:
f2 = torchvision.transforms.Resize(224)
f2(f1(img1)).shape

torch.Size([3, 224, 224])

In [67]:
f = torchvision.transforms.Compose([f1,f2])
model(
    pixel_values=torch.stack([f(img1),f(img2)]),
    labels=torch.tensor([1,1])
)

ImageClassifierOutput(loss=tensor(0.7247, grad_fn=<NllLossBackward0>), logits=tensor([[-0.0370, -0.0565],
        [ 0.0753, -0.0284]], grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)

In [68]:
model.config

ViTConfig {
  "_name_or_path": "google/vit-base-patch16-224-in21k",
  "architectures": [
    "ViTModel"
  ],
  "attention_probs_dropout_prob": 0.0,
  "encoder_stride": 16,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.0,
  "hidden_size": 768,
  "image_size": 224,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "model_type": "vit",
  "num_attention_heads": 12,
  "num_channels": 3,
  "num_hidden_layers": 12,
  "patch_size": 16,
  "problem_type": "single_label_classification",
  "qkv_bias": true,
  "transformers_version": "4.45.2"
}

In [70]:
model.__call__??

Signature: model . __call__ ( * args , ** kwargs ) 
 Docstring: <no docstring>
 Source: 
 def _wrapped_call_impl ( self , * args , ** kwargs ) : 
 if self . _compiled_call_impl is not None : 
 return self . _compiled_call_impl ( * args , ** kwargs ) # type: ignore[misc] 
 else : 
 return self . _call_impl ( * args , ** kwargs ) 
 File: ~/anaconda3/envs/hf/lib/python3.12/site-packages/torch/nn/modules/module.py
 Type: method

In [71]:
from inspect import signature

In [76]:
signature(model.forward)

<Signature (pixel_values: Optional[torch.Tensor] = None, head_mask: Optional[torch.Tensor] = None, labels: Optional[torch.Tensor] = None, output_attentions: Optional[bool] = None, output_hidden_states: Optional[bool] = None, interpolate_pos_encoding: Optional[bool] = None, return_dict: Optional[bool] = None) -> Union[tuple, transformers.modeling_outputs.ImageClassifierOutput]>

In [82]:
model

ViTForImageClassification(
  (vit): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ViTLayer(
          (attention): ViTSdpaAttention(
            (attention): ViTSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linear(in_fe